# 01. Lattice and hopping matrices

This notebook explains how the program numbers sites, assigns hopping amplitudes and builds the normal-state matrix blocks.

Read each explanation together with the code directly below it. The code is original Mathematica; the `.nb` files are the executable versions. Diagrams are existing saved results. These reading notebooks do not rerun the calculation.

## 1. Understand the lattice diagram

### Sites and connections on a square grid

![Sites and connections on a square grid](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/lattice_connections.png)

The circles are lattice sites. Horizontal and vertical arrows indicate axial connections; diagonal arrows indicate connections that change both coordinates. A hopping matrix records these connections as entries between pairs of sites.

The numbers in this illustration are schematic labels. The program below uses its own indexing rule, with the `y` loop inside the `x` loop. Use that rule when translating a coordinate into an array index.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/lattice_connections.png)

## 2. Set the amplitudes

`m = 0.50` controls the scalar hopping dictionary `t`. The entry `t[dx][dy]` is the amplitude assigned to displacement `(dx,dy)`:

| Displacement | Entry | Value when m=0.5 |
| --- | --- | --- |
| Same site `(0,0)` | `3 m/2` | 0.75 |
| One step horizontally or vertically | `-m` | -0.5 |
| One step along each coordinate, diagonally | `m/4` | 0.125 |

`lam = 1` controls the separate spin-mixing dictionary `l`. All four axial entries receive `lam (1-I)`, where `I` is Mathematica's imaginary unit. These values later form `soc`. `SetDirectory` selects the folder used for subsequent file exports; its saved path is machine-specific.

**Hopping parameters** — `BdG topological.nb`, input 1.

```mathematica
(*H1x*)
SetDirectory["\</Users/aditya/Downloads/vortex\>"]
m = 0.50; 
lam = 1;(* next nearest neighbour hopping (diagonal)*)
t[0][0] = 3/2*m;
t[-1][0] = -m;
t[0][-1] = -m;
t[1][0] = -m;
t[0][1] = -m;
t[-1][-1] = m/4;
t[1][-1] = m/4;
t[-1][1] = m/4;
t[1][1] = m/4;
(*SOC spinorbit*)
l[-1][0] = lam(1-I);
l[0][-1] = lam(1-I);
l[1][0] = lam(1-I);
l[0][1] = lam(1-I);

```

## 3. Number the sites

`Nx` and `Ny` are the numbers of sites in the two directions. The loops increment `i` once for each coordinate pair and store that pair in `xpos[i]` and `ypos[i]`. Thus `NN = Nx Ny = 400`.

With Mathematica's one-based indexing, the code gives `i = (x-1) Ny + y`. For `Ny=20`, `(1,1)` is site 1, `(1,20)` is site 20, and `(10,10)` is site 190. `empty` is a 400 × 400 sparse zero matrix. `Vimp` is assigned zero here, but assigning it alone does not insert an impurity into a matrix.

**Coordinates and size** — `BdG topological.nb`, input 1.

```mathematica
Nx = 20;
Ny = 20;
i = 0;

For[ix = 1, ix <= Nx, ix++,
 For[iy = 1, iy <= Ny, iy++, i++;
  xpos[i] = ix;
  ypos[i] = iy;]]
NN = i;

Vimp = 0;
empty= SparseArray[{x_, y_} -> 0, {NN, NN}];

```

## 4. Build the scalar hopping matrix H1

For each pair `(i,j)`, the nested `If` statements inspect coordinate differences with shifts of `-Nx`, `0`, `+Nx` and `-Ny`, `0`, `+Ny`. These shifted copies let a connection cross an edge and enter from the opposite edge: periodic boundaries.

`NumericQ` asks whether the dictionary contains a numerical amplitude for a tested displacement. The first matching branch supplies the entry. `tij[i][j] = tijmn[[j,i]]` transposes the temporary table. The final `SparseArray` keeps numerical entries and makes the unspecified entries zero. The result `H1` has size `NN × NN`.

**Periodic scalar hopping** — `BdG topological.nb`, input 1.

```mathematica
(*Hopping Matrix*)
tijmn = Table[
   If[NumericQ[t[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j] - Ny)]], t[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j] - Ny)],
    If[NumericQ[t[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j])]], t[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j])],
     If[NumericQ[t[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j] + Ny)]], t[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j] + Ny)],
      If[NumericQ[t[xpos[i] - (xpos[j])][ypos[i] - (ypos[j] - Ny)]], t[xpos[i] - (xpos[j])][ypos[i] - (ypos[j] - Ny)],
       If[NumericQ[t[xpos[i] - (xpos[j])][ypos[i] - (ypos[j])]], t[xpos[i] - (xpos[j])][ypos[i] - (ypos[j])],
        If[NumericQ[t[xpos[i] - (xpos[j])][ypos[i] - (ypos[j] + Ny)]], t[xpos[i] - (xpos[j])][ypos[i] - (ypos[j] + Ny)],
         If[NumericQ[t[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j] - Ny)]], t[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j] - Ny)],
          If[NumericQ[t[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j])]], t[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j])],
           If[NumericQ[t[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j] + Ny)]], t[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j] + Ny)]]]]]]]]]],
   {i, 1, NN}, {j, 1, NN}];
For[i = 1, i <= NN, i++,
  For[j = 1, j <= NN, j++,
   tij[i][j] = tijmn[[j, i]]
   ]];
Clear[tijmn]
H1= SparseArray[{x_, y_} /; NumericQ[tij[x][y]] -> tij[x][y], {NN, NN}];

```

## 5. Build the spin-mixing matrix soc

This section repeats the same periodic-image lookup, now using `l` instead of `t`. The result `soc` connects the two spin groups. Its entries are complex because `l` contains `1-I`. `ConjugateTranspose[soc]` means transpose the matrix and reverse the sign of each imaginary part.

**Periodic spin mixing** — `BdG topological.nb`, input 1.

```mathematica
tijmn = Table[
   If[NumericQ[l[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j] - Ny)]], l[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j] - Ny)],
    If[NumericQ[l[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j])]], l[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j])],
     If[NumericQ[l[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j] + Ny)]], l[xpos[i] - (xpos[j] - Nx)][ypos[i] - (ypos[j] + Ny)],
      If[NumericQ[l[xpos[i] - (xpos[j])][ypos[i] - (ypos[j] - Ny)]], l[xpos[i] - (xpos[j])][ypos[i] - (ypos[j] - Ny)],
       If[NumericQ[l[xpos[i] - (xpos[j])][ypos[i] - (ypos[j])]], l[xpos[i] - (xpos[j])][ypos[i] - (ypos[j])],
        If[NumericQ[l[xpos[i] - (xpos[j])][ypos[i] - (ypos[j] + Ny)]], l[xpos[i] - (xpos[j])][ypos[i] - (ypos[j] + Ny)],
         If[NumericQ[l[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j] - Ny)]], l[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j] - Ny)],
          If[NumericQ[l[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j])]], l[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j])],
           If[NumericQ[l[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j] + Ny)]], l[xpos[i] - (xpos[j] + Nx)][ypos[i] - (ypos[j] + Ny)]]]]]]]]]],
   {i, 1, NN}, {j, 1, NN}];
For[i = 1, i <= NN, i++,
  For[j = 1, j <= NN, j++,
   tij[i][j] = tijmn[[j, i]]
   ]];
Clear[tijmn]
soc= SparseArray[{x_, y_} /; NumericQ[tij[x][y]] -> tij[x][y], {NN, NN}];

```

## 6. Read the chemical-potential and block definitions

`Chempot1[mu]` is diagonal with entries `-mu`. Consequently, `H1-Chempot1[mu]` contains `H1 + mu I`.

`Hop1x[mu]` assembles four `NN × NN` blocks into a `2 NN × 2 NN` matrix. Its upper and lower scalar blocks have opposite `H1` signs. **Execution detail:** the iteration does not call `Hop1x`; it separately defines `Huu` and `Hdd`, both with `H1-Chempot1[mu]`. Notebook 02 follows that active assembly.

**Chemical potential and Hop1x** — `BdG topological.nb`, input 1.

```mathematica
Clear[mu]
Chempot1[mu_]:= SparseArray[{x_, x_} ->  -mu, {NN, NN}];
(*Chempot2[mu_]:= SparseArray[{y_, y_} ->  -mu, {NN, NN}];*)
(*Chempot[mu]=ArrayFlatten[{{Chempot1[mu], empty}, {empty,Chempot1[mu]}}];*)
Hop1x[mu_]:= ArrayFlatten[{{H1-Chempot1[mu],soc}, {ConjugateTranspose[soc], -H1-Chempot1[mu]}}];(*adding pauli matrix \[Sigma]x*)
(*HermitianMatrixQ[Hop1x[0]];
MatrixForm[Hop1x[]];*)

```